# ⚡ FranchiseOps AI — Milestone 2
### Enterprise Multi-Agent Franchise Operations Platform


## Step 1 — Install Dependencies


In [7]:
pip install -q streamlit pyngrok bcrypt pyjwt pandas numpy scikit-learn joblib transformers accelerate bitsandbytes plotly streamlit-option-menu faker kaggle lightgbm xgboost kagglehub

## Step 2 — Configure Secrets & Mount Google Drive


In [8]:
import os

def _get_secret(key):
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val: return val
    except Exception:
        pass
    return os.environ.get(key, "")

NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
HF_TOKEN        = _get_secret("HF_TOKEN")
KAGGLE_USERNAME = _get_secret("KAGGLE_USERNAME")
KAGGLE_KEY      = _get_secret("KAGGLE_KEY")
EMAIL_PASSWORD  = _get_secret("EMAIL_PASSWORD")
EMAIL_ID        = _get_secret("EMAIL_ID")
JWT_SECRET_KEY  = _get_secret("JWT_SECRET_KEY") or "franchiseops_ai-dev-secret"
ADMIN_EMAIL     = _get_secret("ADMIN_EMAIL_ID") or "infosys@ai"
ADMIN_PASSWORD  = _get_secret("ADMIN_PASSWORD") or "admin@123"

if KAGGLE_USERNAME: os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
if KAGGLE_KEY:      os.environ["KAGGLE_KEY"]      = KAGGLE_KEY

try:
    if os.path.exists("/content"):
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        STORAGE_DIR = "/content/drive/MyDrive/FranchiseOps_AI"
        print("✅ Google Drive mounted.")
    else:
        STORAGE_DIR = os.path.abspath("./data/FranchiseOps_AI")
except Exception as e:
    STORAGE_DIR = os.path.abspath("./data/FranchiseOps_AI")

os.makedirs(os.path.join(STORAGE_DIR, "models", "hf_cache"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models", "kaggle_cache"), exist_ok=True)
print(f"📁 Storage: {STORAGE_DIR}")
print(f"🔑 HF_TOKEN: {'✅' if HF_TOKEN else '❌ set in Colab Secrets'}")
print(f"🔑 ngrok:    {'✅' if NGROK_AUTHTOKEN else '❌ set in Colab Secrets'}")


Mounted at /content/drive
✅ Google Drive mounted.
📁 Storage: /content/drive/MyDrive/FranchiseOps_AI
🔑 HF_TOKEN: ✅
🔑 ngrok:    ✅


## Step 3 — Verify GPU & Load Qwen-2.5-3B (4-bit NF4)


In [9]:
import os

def _get_secret(key):
    """Read from Colab Secrets first, then environment variable."""
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val: return val
    except Exception:
        pass
    return os.environ.get(key, "")

# ── Load all 7 secrets (set these in Colab Secrets panel) ──────────────────
NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
HF_TOKEN        = _get_secret("HF_TOKEN")
KAGGLE_USERNAME = _get_secret("KAGGLE_USERNAME")
KAGGLE_KEY      = _get_secret("KAGGLE_KEY")
EMAIL_PASSWORD  = _get_secret("EMAIL_PASSWORD")
EMAIL_ID        = _get_secret("EMAIL_ID")
JWT_SECRET_KEY  = _get_secret("JWT_SECRET_KEY") or "franchiseops_ai-dev-secret"
ADMIN_EMAIL     = _get_secret("ADMIN_EMAIL_ID") or "infosys@ai"
ADMIN_PASSWORD  = _get_secret("ADMIN_PASSWORD") or "admin@123"

# Expose Kaggle credentials for the kaggle library
if KAGGLE_USERNAME: os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
if KAGGLE_KEY:      os.environ["KAGGLE_KEY"]      = KAGGLE_KEY

# ── Mount Google Drive (auto-detected in Colab) ─────────────────────────────
try:
    if os.path.exists("/content"):
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        STORAGE_DIR = "/content/drive/MyDrive/FranchiseOps_AI"
        print("✅ Google Drive mounted.")
    else:
        STORAGE_DIR = os.path.abspath("./data/FranchiseOps_AI")
except Exception as e:
    print(f"⚠️  Drive mount skipped ({e}). Using local storage.")
    STORAGE_DIR = os.path.abspath("./data/FranchiseOps_AI")

os.makedirs(STORAGE_DIR, exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models", "kaggle_cache"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models", "hf_cache"), exist_ok=True)

print(f"\n📁 Storage:  {STORAGE_DIR}")
print(f"🔑 JWT:      {'✅ from Colab Secrets' if _get_secret('JWT_SECRET_KEY') else '⚠️  using dev default'}")
print(f"🔑 Admin:    {ADMIN_EMAIL}")
print(f"🔑 HF_TOKEN: {'✅' if HF_TOKEN else '❌ set in Colab Secrets'}")
print(f"🔑 Kaggle:   {'✅' if KAGGLE_KEY else '❌ optional — synthetic fallback'}")
print(f"🔑 ngrok:    {'✅' if NGROK_AUTHTOKEN else '❌ set in Colab Secrets'}")
print(f"🔑 Email:    {'✅' if EMAIL_PASSWORD else '❌ optional'}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted.

📁 Storage:  /content/drive/MyDrive/FranchiseOps_AI
🔑 JWT:      ⚠️  using dev default
🔑 Admin:    infosys@ai
🔑 HF_TOKEN: ✅
🔑 Kaggle:   ❌ optional — synthetic fallback
🔑 ngrok:    ✅
🔑 Email:    ✅


In [10]:
!nvidia-smi


Fri Jul 24 12:28:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto",
)
print("✅ Qwen-2.5-3B loaded. Footprint (GB):", round(model.get_memory_footprint() / 1e9, 2))


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Qwen-2.5-3B loaded. Footprint (GB): 2.01


## Step 4 — Write All Application Modules (`llm_engine`, `config`, `auth`, `db`, `agents`, `dashboard`)


In [12]:
%%writefile llm_engine.py
"""
llm_engine.py — FranchiseOps AI (v4 FINAL — Maximum Speed Edition)
Qwen-2.5-3B-Instruct (4-bit NF4) with:
  • Google Drive Persistent Caching (hf_cache) — instant reload without re-download
  • low_cpu_mem_usage=True + attn_implementation="sdpa" (falls back to "eager") — faster load AND faster generation on T4
  • torch.inference_mode() + use_cache=True + greedy decode — ~1 sec responses
  • Single-Pass generate_debate_and_synthesis() — all 3 agents + synthesis in ~1.5 sec
  • Trimmed max_new_tokens across all 3 generation functions for lower per-call latency
"""
import os, json, re, torch, threading
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from config import HF_TOKEN

MODEL_ID  = "Qwen/Qwen2.5-3B-Instruct"
CACHE_DIR = "/content/drive/MyDrive/FranchiseOps_AI/models/hf_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

_model     = None
_tokenizer = None
_load_lock = threading.Lock()


def get_model():
    global _model, _tokenizer
    if _model is not None:
        return _model, _tokenizer
    with _load_lock:
        if _model is not None:          # someone else finished loading while we waited
            return _model, _tokenizer
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        kw = {"token": HF_TOKEN, "cache_dir": CACHE_DIR} if HF_TOKEN else {"cache_dir": CACHE_DIR}
        _tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **kw)
        # sdpa (PyTorch's built-in scaled-dot-product-attention kernel) generates
        # noticeably faster than "eager" on T4 -- eager only wins on load time.
        # Fall back to eager automatically if this transformers/torch combo
        # doesn't support sdpa for Qwen2, so this never becomes a new crash.
        try:
            _model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID,
                quantization_config=bnb,
                device_map="auto",
                torch_dtype=torch.float16,
                low_cpu_mem_usage=True,
                attn_implementation="sdpa",
                **kw,
            )
        except Exception:
            _model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID,
                quantization_config=bnb,
                device_map="auto",
                torch_dtype=torch.float16,
                low_cpu_mem_usage=True,
                attn_implementation="eager",
                **kw,
            )
        _model.eval()
    return _model, _tokenizer


def warmup_llm():
    """Load model into GPU memory for instant subsequent generation."""
    try:
        get_model()
        return _model is not None
    except Exception:
        return False


def is_llm_loaded():
    return _model is not None


_warmup_thread_started = False

def start_background_warmup():
    """
    Kicks off model loading in a background thread exactly once per process,
    called at app.py import time. This way the model is already warm -- or
    already warming up -- before anyone opens the AI Copilot tab, instead of
    blocking on someone's first click mid-demo. get_model()'s _load_lock means
    a manual warmup_llm() call or a real chat request made while this thread
    is still loading just waits for it, rather than starting a second,
    duplicate (and GPU-memory-doubling) load.
    """
    global _warmup_thread_started
    if _warmup_thread_started:
        return
    _warmup_thread_started = True
    threading.Thread(target=warmup_llm, daemon=True).start()


def _run(msgs, max_tokens=100, greedy=True):
    """Core low-overhead generation helper."""
    model, tok = get_model()
    tmpl   = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(tmpl, return_tensors="pt").to(model.device)
    gen_kw = dict(
        max_new_tokens=max_tokens,
        use_cache=True,
        pad_token_id=tok.eos_token_id,
        eos_token_id=tok.eos_token_id,
    )
    if greedy:
        gen_kw["do_sample"] = False
    else:
        gen_kw["do_sample"]   = True
        gen_kw["temperature"] = 0.2
        gen_kw["top_p"]       = 0.9
    with torch.inference_mode():
        out = model.generate(**inputs, **gen_kw)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


def generate_json(prompt, schema_keys=None):
    """Returns a structured JSON dict from the model — greedy, minimal tokens."""
    sys_p = "You are an AI franchise intelligence engine. Respond ONLY with a valid JSON object."
    if schema_keys:
        sys_p += f" Required keys: {', '.join(schema_keys)}."
    raw = _run(
        [{"role": "system", "content": sys_p}, {"role": "user", "content": prompt}],
        max_tokens=150,
        greedy=True,
    )
    def _repair_json(text):
        text = re.sub(r'```json\s*|\s*```', '', text)
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m: text = m.group(0)
        # Fix missing commas between key-value pairs (e.g. "val"\n"key": or "val" "key":)
        text = re.sub(r'(["]|\d|true|false)\s*\n\s*(["\w]+":)', r'\1,\n\2', text)
        text = re.sub(r'(["]|\d|true|false)\s+(["\w]+":)', r'\1, \2', text)
        # Fix trailing commas before closing brace
        text = re.sub(r',\s*\}', '}', text)
        return text

    try:
        return json.loads(_repair_json(raw))
    except Exception:
        if schema_keys:
            # Fallback regex extraction of key-value pairs if strict JSON still fails
            out = {}
            for k in schema_keys:
                km = re.search(rf'"{k}"\s*:\s*"([^"]*)"|"{k}"\s*:\s*([^,\}}]+)', raw)
                if km: out[k] = (km.group(1) if km.group(1) is not None else km.group(2)).strip()
                else: out[k] = "N/A"
            if any(v != "N/A" for v in out.values()): return out
        return {"error": "JSON parse failed", "raw": raw}


# ── Agent Roles ───────────────────────────────────────────────────────────────
AGENT_ROLES = {
    "agent1": ("Workforce Retention Agent",
               "You specialise in employee satisfaction, overtime fatigue, and attrition risk."),
    "agent2": ("Outlet Territory Clustering Agent",
               "You specialise in store revenue vs cost clustering, headcount efficiency, tier rating."),
    "agent3": ("Supply Chain & Inventory Advisor Agent",
               "You specialise in weather-driven demand surges, SKU stockout probabilities, lead times."),
}


def generate_debate_and_synthesis(user_query, agent1_context, agent2_context, agent3_context, db_stats=None):
    """
    Single-pass structured generation — outputs Agent 1 / 2 / 3 views + Synthesis.
    Target latency: ~2 sec on T4.
    """
    system_prompt = (
        "You are the FranchiseOps AI Multi-Agent Engine. "
        "Analyze the query and all data. Reply STRICTLY in this format:\n"
        "[AGENT 1]: <1 bullet on workforce/attrition>\n"
        "[AGENT 2]: <1 bullet on outlet clustering/revenue>\n"
        "[AGENT 3]: <1 bullet on inventory/weather>\n"
        "[SYNTHESIS]: <2 sentences executive recommendation>"
    )
    ctx = (
        f"QUERY: {user_query}\n"
        f"A1: {json.dumps(agent1_context)}\n"
        f"A2: {json.dumps(agent2_context)}\n"
        f"A3: {json.dumps(agent3_context)}"
    )
    if db_stats:
        ctx += f"\nDB: {json.dumps(db_stats)}"

    raw = _run(
        [{"role": "system", "content": system_prompt}, {"role": "user", "content": ctx}],
        max_tokens=100,
        greedy=True,
    )
    res = {
        "agent1": "Overtime hours and low satisfaction are primary attrition drivers.",
        "agent2": "Outlet clustering identifies underperforming stores with high cost ratios.",
        "agent3": "Weather-driven demand surges are causing critical SKU stockout risk.",
        "synthesis": raw,
    }
    try:
        for key, tag, nxt in [
            ("agent1", "AGENT 1", "AGENT 2"),
            ("agent2", "AGENT 2", "AGENT 3"),
            ("agent3", "AGENT 3", "SYNTHESIS"),
        ]:
            m = re.search(rf"\[{tag}\]:\s*(.*?)(?=\[{nxt}\]|\Z)", raw, re.DOTALL | re.IGNORECASE)
            if m:
                res[key] = m.group(1).strip()
        m = re.search(r"\[SYNTHESIS\]:\s*(.*)", raw, re.DOTALL | re.IGNORECASE)
        if m:
            res["synthesis"] = m.group(1).strip()
    except Exception:
        pass
    return res


def orchestrate_3_agents_query(user_question, agent1_context, agent2_context, agent3_context, db_stats=None):
    """Fast greedy single-pass answer — target latency ~1.5 sec on T4."""
    sys_p = (
        "You are FranchiseOps AI Orchestrator. "
        "Give a crisp 2-sentence actionable executive answer using all agent data."
    )
    ctx = (
        f"QUERY: {user_question}\n"
        f"A1: {json.dumps(agent1_context)}\n"
        f"A2: {json.dumps(agent2_context)}\n"
        f"A3: {json.dumps(agent3_context)}"
    )
    if db_stats:
        ctx += f"\nDB: {json.dumps(db_stats)}"
    return _run(
        [{"role": "system", "content": sys_p}, {"role": "user", "content": ctx}],
        max_tokens=90,
        greedy=True,
    )


Writing llm_engine.py


In [13]:
%%writefile config.py
"""
config.py — FranchiseOps AI (v3 FINAL)
All secrets from Colab userdata. KMEANS_MODEL_PATH = kmeans_outlets.joblib (spec compliant).
"""
import os

def _get_secret(key):
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val: return val
    except Exception:
        pass
    return os.environ.get(key, "")

try:
    from __main__ import (STORAGE_DIR, NGROK_AUTHTOKEN, HF_TOKEN,
                          KAGGLE_USERNAME, KAGGLE_KEY, EMAIL_PASSWORD,
                          ADMIN_EMAIL, ADMIN_PASSWORD, EMAIL_ID)
except ImportError:
    STORAGE_DIR    = ("/content/drive/MyDrive/FranchiseOps_AI"
                      if os.path.exists("/content/drive/MyDrive") else
                      os.path.abspath("./data/FranchiseOps_AI"))
    NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
    NGROK_AUTH_TOKEN = NGROK_AUTHTOKEN # Alias for launch cell compatibility
    HF_TOKEN        = _get_secret("HF_TOKEN")
    KAGGLE_USERNAME = _get_secret("KAGGLE_USERNAME")
    KAGGLE_KEY      = _get_secret("KAGGLE_KEY")
    EMAIL_PASSWORD  = _get_secret("EMAIL_PASSWORD")
    EMAIL_ID        = _get_secret("EMAIL_ID")
    JWT_SECRET_KEY  = _get_secret("JWT_SECRET_KEY") or "franchiseops-dev-secret-changeme"
    ADMIN_EMAIL     = _get_secret("ADMIN_EMAIL_ID")  or "infosys@ai"
    ADMIN_PASSWORD  = _get_secret("ADMIN_PASSWORD")  or "admin@123"

os.makedirs(STORAGE_DIR, exist_ok=True)
DB_PATH          = os.path.join(STORAGE_DIR, "franchiseops.db")
MODELS_DIR       = os.path.join(STORAGE_DIR, "models")
KAGGLE_CACHE_DIR = os.path.join(MODELS_DIR, "kaggle_cache")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(KAGGLE_CACHE_DIR, exist_ok=True)

# Model paths (filenames match Infosys spec exactly)
AGENT1_MODEL_PATH = os.path.join(MODELS_DIR, "attrition_lr.joblib")
KMEANS_MODEL_PATH = os.path.join(MODELS_DIR, "kmeans_outlets.joblib")   # spec: kmeans_outlets
AGENT2_MODEL_PATH = KMEANS_MODEL_PATH                                    # alias
AGENT2_REG_PATH   = os.path.join(MODELS_DIR, "revenue_rf.joblib")
AGENT3_MODEL_PATH = os.path.join(MODELS_DIR, "inventory_demand_gb.joblib")


Writing config.py


In [14]:
%%writefile ui_theme.py
"""
Shared ui_theme.py for FreightQuote AI & FranchiseOps AI
Exact Neo-Brutalist UI styling, layout cards, and status badges.
"""
import streamlit as st

COLORS = {
    "bg_main":       "#ffffff",
    "bg_card":       "#ffffff",
    "bg_alt":        "#f2f4f6",
    "text_heading":  "#272343",
    "text_body":     "#2d334a",
    "text_main":     "#2d334a",
    "text_muted":    "#626880",
    "border":        "#272343",
    "accent":        "#ffd803",
    "accent_subtle": "#ffe866",
    "accent_text":   "#272343",
    "cyan":          "#e3f6f5",
    "pink":          "#ffd3e2",
    "green":         "#34d399",
    "yellow":        "#fbbf24",
    "red":           "#f87171",
}

NEO_BRUTALIST_CSS = f"""
<style>
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&family=Space+Grotesk:wght@600;700&family=JetBrains+Mono:wght@500;700&display=swap');

html, body, [class*="css"] {{
    font-family: 'Plus Jakarta Sans', sans-serif;
    color: {COLORS["text_body"]};
    background-color: {COLORS["bg_main"]};
}}

h1, h2, h3, h4, h5, h6 {{
    font-family: 'Space Grotesk', sans-serif;
    color: {COLORS["text_heading"]};
    font-weight: 700;
}}

.pn-card {{
    background: {COLORS["bg_card"]};
    border: 3px solid {COLORS["border"]};
    border-radius: 12px;
    padding: 20px;
    margin-bottom: 20px;
    box-shadow: 6px 6px 0px {COLORS["border"]};
    transition: transform 0.15s ease, box-shadow 0.15s ease;
}}
.pn-card:hover {{
    transform: translate(-2px, -2px);
    box-shadow: 8px 8px 0px {COLORS["border"]};
}}
.pn-card-alt {{
    background: {COLORS["cyan"]};
    border: 3px solid {COLORS["border"]};
    border-radius: 12px;
    padding: 20px;
    margin-bottom: 20px;
    box-shadow: 6px 6px 0px {COLORS["border"]};
}}

.pn-badge {{
    display: inline-block;
    padding: 4px 12px;
    border: 2px solid {COLORS["border"]};
    border-radius: 6px;
    font-family: 'JetBrains Mono', monospace;
    font-weight: 700;
    font-size: 13px;
    box-shadow: 2px 2px 0px {COLORS["border"]};
    text-transform: uppercase;
}}
.agent-badge {{
    display: inline-block;
    padding: 4px 14px;
    background: {COLORS["accent"]};
    color: {COLORS["text_heading"]};
    border: 2px solid {COLORS["border"]};
    border-radius: 8px;
    font-family: 'Space Grotesk', sans-serif;
    font-weight: 700;
    font-size: 14px;
    box-shadow: 3px 3px 0px {COLORS["border"]};
}}

/* Streamlit Buttons Matching Login Portal */
div.stButton > button {{
    background: #ffd803 !important;
    color: #272343 !important;
    font-family: 'Space Grotesk', sans-serif !important;
    font-weight: 700 !important;
    border: 3px solid #272343 !important;
    border-radius: 10px !important;
    padding: 10px 22px !important;
    box-shadow: 4px 4px 0px #272343 !important;
    transition: all 0.15s ease !important;
}}
div.stButton > button:hover {{
    transform: translate(-2px, -2px) !important;
    box-shadow: 6px 6px 0px #272343 !important;
    background: #ffe866 !important;
}}

/* Streamlit Inputs & Selectboxes Matching Login Portal */
div[data-baseweb="input"] > div, div[data-baseweb="select"] > div {{
    background: #ffffff !important;
    border: 3px solid #272343 !important;
    border-radius: 8px !important;
    box-shadow: 3px 3px 0px #272343 !important;
}}

/* Streamlit Tabs Matching Login Portal */
button[data-baseweb="tab"] {{
    font-family: 'Space Grotesk', sans-serif !important;
    font-weight: 700 !important;
    color: #2d334a !important;
}}
button[data-baseweb="tab"][aria-selected="true"] {{
    color: #272343 !important;
    border-bottom: 3px solid #ffd803 !important;
}}
</style>
"""

def inject_css():
    st.markdown(NEO_BRUTALIST_CSS, unsafe_allow_html=True)

def apply_theme():
    inject_css()

def render_header(title, subtitle="", icon="⚡"):
    inject_css()
    st.markdown(f"""
    <div style="background:{COLORS['bg_card']};border:3px solid {COLORS['border']};border-radius:14px;padding:22px 28px;margin-bottom:24px;box-shadow:6px 6px 0px {COLORS['border']};">
        <div style="display:flex;align-items:center;gap:16px;">
            <div style="font-size:42px;line-height:1;">{icon}</div>
            <div>
                <h1 style="margin:0;font-size:26px;letter-spacing:-0.5px;">{title}</h1>
                <p style="margin:4px 0 0;color:{COLORS['text_muted']};font-size:14px;">{subtitle}</p>
            </div>
        </div>
    </div>
    """, unsafe_allow_html=True)

def render_card(content, alt=False):
    c_class = "pn-card-alt" if alt else "pn-card"
    st.markdown(f'<div class="{c_class}">{content}</div>', unsafe_allow_html=True)

def risk_badge(text, level="Low"):
    color_map = {"Low": COLORS["green"], "Medium": COLORS["yellow"], "High": COLORS["red"], "Critical": COLORS["red"]}
    c = color_map.get(level, COLORS["cyan"])
    return f'<span class="pn-badge" style="background:{c};">{text}</span>'


Writing ui_theme.py


In [15]:
%%writefile auth.py
"""
FranchiseOps AI - auth.py
Standardized SQLite authentication system matching Login_Page (1).ipynb.
Supports Login, Register (with Enterprise Roles), Forgot Password (security question check), and JWT tokens.
"""
import sqlite3, jwt, bcrypt, datetime, streamlit as st
try:
    from config import DB_PATH, JWT_SECRET_KEY, ADMIN_EMAIL, ADMIN_PASSWORD
    JWT_SECRET = JWT_SECRET_KEY
except (ImportError, AttributeError):
    from config import DB_PATH
    JWT_SECRET = "super-secret-franchiseops-key-2026"
    ADMIN_EMAIL = "infosys@ai"
    ADMIN_PASSWORD = "admin@123"
from ui_theme import COLORS

def get_conn():
    return sqlite3.connect(DB_PATH, check_same_thread=False)

def hash_txt(t):
    return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()

def check_txt(t, h):
    try: return bcrypt.checkpw(t.encode(), h.encode()) if h else False
    except: return False

def make_jwt(email, username, role):
    return jwt.encode({"email": email, "username": username, "role": role, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=6)}, JWT_SECRET, algorithm="HS256")

def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except: return None

def lock_account(user_id):
    with get_conn() as conn:
        # Lock for 1 hour
        lock_until = datetime.datetime.now() + datetime.timedelta(hours=1)
        conn.execute("UPDATE users SET account_status='locked', lock_until=? WHERE id=?", (lock_until, user_id))
        conn.commit()

def unlock_account(user_id):
    with get_conn() as conn:
        conn.execute("UPDATE users SET failed_attempts=0, lock_until=NULL, account_status='active' WHERE id=?", (user_id,))
        conn.commit()

@st.cache_resource
def init_auth():
    with get_conn() as conn:
        conn.execute("""CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE,
            email TEXT UNIQUE,
            password_hash TEXT,
            security_question TEXT,
            security_answer_hash TEXT,
            role TEXT DEFAULT 'Franchisee',
            failed_attempts INTEGER DEFAULT 0,
            lock_until TIMESTAMP,
            account_status TEXT DEFAULT 'active',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )""")
        # Alter table statements for backward compatibility if columns don't exist
        try: conn.execute("ALTER TABLE users ADD COLUMN security_question TEXT")
        except Exception: pass
        try: conn.execute("ALTER TABLE users ADD COLUMN security_answer_hash TEXT")
        except Exception: pass
        try: conn.execute("ALTER TABLE users ADD COLUMN failed_attempts INTEGER DEFAULT 0")
        except Exception: pass
        try: conn.execute("ALTER TABLE users ADD COLUMN lock_until TIMESTAMP")
        except Exception: pass
        try: conn.execute("ALTER TABLE users ADD COLUMN account_status TEXT DEFAULT 'active'")
        except Exception: pass

        # Seed default admin account if not exists
        if not conn.execute("SELECT id FROM users WHERE email=?", (ADMIN_EMAIL,)).fetchone():
            conn.execute("""INSERT OR IGNORE INTO users
                         (username, email, password_hash, security_question, security_answer_hash, role, account_status)
                         VALUES (?, ?, ?, ?, ?, ?, ?)""",
                         ("Administrator", ADMIN_EMAIL, hash_txt(ADMIN_PASSWORD), "What is your pet name?", hash_txt("admin"), "Admin", "active"))
            conn.commit()

def render_auth_portal():
    init_auth()
    if "token" not in st.session_state: st.session_state["token"] = None
    if "auth_tab" not in st.session_state: st.session_state["auth_tab"] = "Login"

    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:44px;margin-bottom:8px;">⚡</div>
        <h1 style="font-size:2rem !important;margin:0;">FranchiseOps AI Portal</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">Enterprise Multi-Agent Franchise Intelligence System</p>
    </div>
    """, unsafe_allow_html=True)

    c1, c2, c3 = st.columns([1, 2, 1])
    with c2:
        tab1, tab2, tab3 = st.tabs(["🔐 Sign In", "📝 Register Account", "🔑 Reset Password"])

        with tab1:
            login_email = st.text_input("Email / Username", key="l_email", placeholder=ADMIN_EMAIL)
            login_pw = st.text_input("Password", type="password", key="l_pw", placeholder="••••••••")
            if st.button("🚀 Sign In to Portal", key="btn_login"):
                with get_conn() as conn:
                    user = conn.execute("SELECT id, username, email, password_hash, role, failed_attempts, lock_until, account_status FROM users WHERE email=? OR username=?", (login_email, login_email)).fetchone()
                    if user:
                        user_id, username, email, password_hash, role, failed_attempts, lock_until, account_status = user

                        if account_status == 'locked' and lock_until and datetime.datetime.now() < datetime.datetime.strptime(lock_until, '%Y-%m-%d %H:%M:%S.%f'):
                            st.error(f"Account is locked. Please try again after {datetime.datetime.strptime(lock_until, '%Y-%m-%d %H:%M:%S.%f').strftime('%H:%M')}.")
                            st.stop()

                        if check_txt(login_pw, password_hash):
                            conn.execute("UPDATE users SET failed_attempts=0, lock_until=NULL, account_status='active' WHERE id=?", (user_id,))
                            conn.commit()
                            st.session_state["token"] = make_jwt(email, username, role)
                            st.session_state["username"] = username
                            st.session_state["role"] = role
                            st.success(f"Welcome back, {username} [{role}]!")
                            st.rerun()
                        else:
                            failed_attempts += 1
                            conn.execute("UPDATE users SET failed_attempts=? WHERE id=?", (failed_attempts, user_id))
                            conn.commit()
                            if failed_attempts >= 3:
                                lock_account(user_id)
                                st.error("Too many failed attempts. Account locked for 1 hour.")
                                st.stop()
                            else:
                                st.error(f"Invalid email/username or password. {3 - failed_attempts} attempts remaining.")
                    else:
                        st.error("Invalid email/username or password.")

        with tab2:
            r_user = st.text_input("Username", key="r_u")
            r_email = st.text_input("Email Address", key="r_e")
            r_pw = st.text_input("Create Password", type="password", key="r_p")
            r_role = st.selectbox("Select Enterprise Role", ["Franchise Owner", "Regional Operations Manager", "Store Manager", "Supply Chain Analyst"], key="r_role")
            r_q = st.selectbox("Security Question", ["What is your pet name?", "What city were you born in?", "What is your favorite school teacher's name?"], key="r_q")
            r_a = st.text_input("Security Answer", key="r_a")
            if st.button("✨ Create Franchisee Account", key="btn_reg"):
                if r_user and r_email and r_pw and r_a:
                    try:
                        with get_conn() as conn:
                            conn.execute("INSERT INTO users (username, email, password_hash, security_question, security_answer_hash, role, account_status, failed_attempts) VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
                                         (r_user, r_email, hash_txt(r_pw), r_q, hash_txt(r_a.lower().strip()), r_role, 'active', 0))
                            conn.commit()
                        st.success(f"Account registered with role [{r_role}]! Please switch to Sign In tab.")
                    except Exception as e:
                        st.error(f"Registration failed: Email or username may already exist. {e}")
                else:
                    st.warning("Please fill out all fields.")

        with tab3:
            f_email = st.text_input("Registered Email", key="f_e")
            if st.button("Verify Email & Fetch Question", key="btn_f1"):
                with get_conn() as conn:
                    u = conn.execute("SELECT security_question FROM users WHERE email=?", (f_email,)).fetchone()
                if u:
                    st.session_state["reset_email"] = f_email
                    st.session_state["reset_q"] = u[0]
                    st.rerun()
                else:
                    st.error("Email not found.")

            if st.session_state.get("reset_email"):
                st.info(f"Security Question: **{st.session_state.get('reset_q')}**")
                ans_try = st.text_input("Enter Answer", key="f_ans")
                new_pw = st.text_input("New Password", type="password", key="f_npw")
                if st.button("Confirm Password Reset", key="btn_f2"):
                    with get_conn() as conn:
                        u_hash = conn.execute("SELECT security_answer_hash FROM users WHERE email=?", (st.session_state["reset_email"],)).fetchone()
                    if u_hash and check_txt(ans_try.lower().strip(), u_hash[0]):
                        with get_conn() as conn:
                            conn.execute("UPDATE users SET password_hash=?, failed_attempts=0, lock_until=NULL, account_status='active' WHERE email=?", (hash_txt(new_pw), st.session_state["reset_email"]))
                            conn.commit()
                        st.success("Password reset successfully! Please sign in.")
                        st.session_state["reset_email"] = None
                    else:
                        st.error("Incorrect security answer.")

Overwriting auth.py


In [16]:
%%writefile db.py
import sqlite3
from config import DB_PATH

def get_conn():
    return sqlite3.connect(DB_PATH, check_same_thread=False)

def init_db():
    with get_conn() as conn:
        conn.execute("""CREATE TABLE IF NOT EXISTS outlets (
            outlet_id TEXT PRIMARY KEY, outlet_name TEXT, city TEXT,
            monthly_revenue REAL, monthly_costs REAL, staff_headcount INTEGER,
            avg_overtime_hours REAL, customer_satisfaction REAL,
            tier_cluster TEXT, attrition_risk_level TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS staff (
            staff_id TEXT PRIMARY KEY, outlet_id TEXT, employee_name TEXT,
            role TEXT, monthly_salary REAL, weekly_overtime_hrs REAL,
            job_satisfaction INTEGER, employee_age INTEGER, tenure_years REAL,
            work_life_balance INTEGER, predicted_attrition_prob REAL,
            intervention_status TEXT DEFAULT 'Active')""")
        conn.execute("""CREATE TABLE IF NOT EXISTS inventory_records (
            record_id INTEGER PRIMARY KEY AUTOINCREMENT, outlet_id TEXT,
            sku_name TEXT, current_stock INTEGER, weekly_demand INTEGER,
            reorder_threshold INTEGER, stockout_risk_prob REAL,
            last_updated TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS merged_datasets (
            id INTEGER PRIMARY KEY AUTOINCREMENT, agent_target TEXT, dataset_source TEXT,
            outlet_id TEXT, employee_age INTEGER, overtime_hours REAL,
            job_satisfaction INTEGER, attrition_target INTEGER, monthly_sales_usd REAL,
            operating_cost_usd REAL, tier_cluster_label INTEGER, sku_demand INTEGER,
            weather_impact_factor REAL, stockout_target INTEGER,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        # Updated users table schema with failed_attempts, lock_until, and account_status
        conn.execute("""CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE,
            email TEXT UNIQUE,
            password_hash TEXT,
            security_question TEXT,
            security_answer_hash TEXT,
            role TEXT DEFAULT 'Franchisee',
            failed_attempts INTEGER DEFAULT 0,
            lock_until TIMESTAMP,
            account_status TEXT DEFAULT 'active',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )""")
        # Alter table statements for backward compatibility if columns don't exist
        try: conn.execute("ALTER TABLE users ADD COLUMN security_question TEXT")
        except Exception: pass
        try: conn.execute("ALTER TABLE users ADD COLUMN security_answer_hash TEXT")
        except Exception: pass
        try: conn.execute("ALTER TABLE users ADD COLUMN failed_attempts INTEGER DEFAULT 0")
        except Exception: pass
        try: conn.execute("ALTER TABLE users ADD COLUMN lock_until TIMESTAMP")
        except Exception: pass
        try: conn.execute("ALTER TABLE users ADD COLUMN account_status TEXT DEFAULT 'active'")
        except Exception: pass
        conn.execute("""CREATE TABLE IF NOT EXISTS ml_models (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            agent_name TEXT, model_name TEXT, r2_score REAL,
            rmse REAL, accuracy REAL, training_rows INTEGER,
            file_path TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS notifications (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            channel TEXT, recipient TEXT, subject TEXT, message TEXT,
            status TEXT DEFAULT 'Sent',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS chat_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT NOT NULL, role TEXT NOT NULL, content TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.commit()

def save_ml_metrics(agent_name, model_name, r2, rmse, acc, rows, path):
    with get_conn() as conn:
        conn.execute("INSERT INTO ml_models "
                     "(agent_name,model_name,r2_score,rmse,accuracy,training_rows,file_path) "
                     "VALUES (?,?,?,?,?,?,?)",
                     (agent_name, model_name, r2, rmse, acc, rows, path))
        conn.commit()

def load_chat_history(username, conn_fn=None, limit=60):
    fn = conn_fn or get_conn
    with fn() as conn:
        rows = conn.execute(
            "SELECT role,content FROM chat_history WHERE username=? "
            "ORDER BY id DESC LIMIT ?", (username, limit)).fetchall()
    return [{"role":r[0],"content":r[1]} for r in reversed(rows)]

def save_chat_message(username, role, content, conn_fn=None):
    fn = conn_fn or get_conn
    with fn() as conn:
        conn.execute("INSERT INTO chat_history (username,role,content) VALUES (?,?,?)",
                     (username, role, content))
        conn.commit()

def clear_chat_history(username, conn_fn=None):
    fn = conn_fn or get_conn
    with fn() as conn:
        conn.execute("DELETE FROM chat_history WHERE username=?", (username,))
        conn.commit()

Overwriting db.py


In [17]:
%%writefile weather_context.py
"""
weather_context.py for FranchiseOps AI
Simulates local Indian city weather disruptions and logistics delays across franchise outlets.
"""
import random

CITY_WEATHER_REPORTS = {
    "Mumbai (MH)": {"status": "Heavy Monsoon Rain & Waterlogging", "temp_c": 28, "demand_impact_pct": -18.0, "supply_delay_days": 2, "attrition_stress": "High"},
    "Bengaluru (KA)": {"status": "Pleasant / Light Showers", "temp_c": 24, "demand_impact_pct": 12.0, "supply_delay_days": 0, "attrition_stress": "Normal"},
    "Delhi NCR (DL)": {"status": "Intense Summer Heatwave & Smog", "temp_c": 42, "demand_impact_pct": 15.0, "supply_delay_days": 1, "attrition_stress": "High"},
    "Hyderabad (TG)": {"status": "Clear & Warm", "temp_c": 33, "demand_impact_pct": 8.0, "supply_delay_days": 0, "attrition_stress": "Normal"},
    "Chennai (TN)": {"status": "Humid & Coastal Showers", "temp_c": 35, "demand_impact_pct": -5.0, "supply_delay_days": 1, "attrition_stress": "Medium"},
    "Pune (MH)": {"status": "Cloudy & Breezy", "temp_c": 26, "demand_impact_pct": 10.0, "supply_delay_days": 0, "attrition_stress": "Normal"},
    "Ahmedabad (GJ)": {"status": "Dry & High Heat", "temp_c": 40, "demand_impact_pct": -8.0, "supply_delay_days": 1, "attrition_stress": "Medium"},
    "Kolkata (WB)": {"status": "Thunderstorms & High Humidity", "temp_c": 32, "demand_impact_pct": -12.0, "supply_delay_days": 2, "attrition_stress": "High"}
}

def get_city_weather(city_name):
    for k, v in CITY_WEATHER_REPORTS.items():
        if k.lower() in city_name.lower() or city_name.lower() in k.lower():
            return {"city": k, **v}
    return {"city": city_name, "status": "Fair Weather Conditions", "temp_c": 30, "demand_impact_pct": 0.0, "supply_delay_days": 0, "attrition_stress": "Normal"}

def get_weather_report(port_name):
    return {"port": port_name, "status": "Normal Marine Conditions", "temp_c": 25, "wind_kt": 15, "delay_penalty_multiplier": 1.00}


Writing weather_context.py


In [18]:
%%writefile notifications.py
"""
FranchiseOps AI - notifications.py
Multi-channel alert center simulating SMS, Email, and In-App notifications stored in SQLite.
"""
from db import get_conn

def send_alert(channel, recipient, subject, message):
    with get_conn() as conn:
        conn.execute("INSERT INTO notifications (channel, recipient, subject, message, status) VALUES (?, ?, ?, ?, ?)",
                     (channel, recipient, subject, message, "Delivered"))
        conn.commit()
    print(f"[{channel.upper()}] To: {recipient} | Subject: {subject} | Status: Delivered")

def get_recent_alerts(limit=15):
    with get_conn() as conn:
        return conn.execute("SELECT id, channel, recipient, subject, message, created_at FROM notifications ORDER BY id DESC LIMIT ?", (limit,)).fetchall()


Writing notifications.py


In [19]:
%%writefile seed_data.py
"""
FranchiseOps AI - seed_data.py
Pre-seeds the database with realistic outlets, staff members, shift logs, and inventory benchmarks.
"""
from db import get_conn, init_db
from notifications import send_alert

def seed_all():
    init_db()
    with get_conn() as conn:
        # Seed Outlets
        if not conn.execute("SELECT count(*) FROM outlets").fetchone()[0]:
            outlets = [
                ("OUT-101", "Mumbai Flagship Store", "Mumbai (MH)", 145000, 112000, 24, 18.5, 4.2, "Tier 3 (At-Risk)", "High Attrition"),
                ("OUT-102", "Bengaluru Tech Hub Cafe", "Bengaluru (KA)", 285000, 165000, 32, 4.2, 4.8, "Tier 1 (Apex)", "Low Attrition"),
                ("OUT-103", "Delhi NCR Metro Express", "Delhi NCR (DL)", 210000, 155000, 28, 14.0, 4.5, "Tier 2 (Stable)", "Moderate Attrition"),
                ("OUT-104", "Hyderabad Central Hub", "Hyderabad (TG)", 125000, 118000, 18, 22.0, 3.8, "Tier 3 (At-Risk)", "Critical Attrition"),
                ("OUT-105", "Chennai Coastal Kiosk", "Chennai (TN)", 195000, 138000, 26, 6.5, 4.7, "Tier 1 (Apex)", "Low Attrition"),
                ("OUT-106", "Pune IT Park Outlet", "Pune (MH)", 172000, 129000, 22, 9.8, 4.4, "Tier 2 (Stable)", "Low Attrition"),
            ]
            conn.executemany("INSERT INTO outlets VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, CURRENT_TIMESTAMP)", outlets)

        # Seed Staff
        if not conn.execute("SELECT count(*) FROM staff").fetchone()[0]:
            staff = [
                ("ST-5001", "OUT-101", "Marcus Vance", "Shift Supervisor", 3920.0, 21.0, 2, 32, 4.5, 2, 0.82, "Retention Bonus Offered"),
                ("ST-5002", "OUT-101", "Elena Rostova", "Barista / Cashier", 2880.0, 19.5, 2, 26, 2.0, 2, 0.79, "Schedule Adjusted"),
                ("ST-5003", "OUT-102", "David Chen", "Store Manager", 5120.0, 3.5, 5, 41, 8.5, 4, 0.12, "Stable"),
                ("ST-5004", "OUT-104", "Samantha Diaz", "Kitchen Lead", 3360.0, 24.5, 1, 29, 3.0, 1, 0.89, "Immediate Review Required"),
                ("ST-5005", "OUT-105", "James Wilson", "Team Lead", 4000.0, 5.0, 4, 36, 6.0, 3, 0.18, "Stable"),
            ]
            conn.executemany("INSERT INTO staff (staff_id, outlet_id, employee_name, role, monthly_salary, weekly_overtime_hrs, job_satisfaction, employee_age, tenure_years, work_life_balance, predicted_attrition_prob, intervention_status) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", staff)

        # Seed Inventory
        if not conn.execute("SELECT count(*) FROM inventory_records").fetchone()[0]:
            inventory = [
                ("OUT-101", "Premium Coffee Beans (Kg)", 140, 320, 180, 0.84),
                ("OUT-101", "Organic Milk Syrups (L)", 85, 190, 100, 0.78),
                ("OUT-102", "Premium Coffee Beans (Kg)", 580, 450, 250, 0.12),
                ("OUT-104", "Eco-Packaging Cups (Box)", 40, 210, 150, 0.91),
                ("OUT-105", "Artisan Tea Blends (Kg)", 310, 220, 140, 0.15),
            ]
            conn.executemany("INSERT INTO inventory_records (outlet_id, sku_name, current_stock, weekly_demand, reorder_threshold, stockout_risk_prob) VALUES (?, ?, ?, ?, ?, ?)", inventory)
            conn.commit()

    send_alert("Email", "franchisee@franchiseops.ai", "Franchise Operations Initialized", "Database seeded with 6 regional outlets, staff logs, and inventory benchmarks.")
    print("✅ Database pre-seeded successfully.")


Writing seed_data.py


In [20]:
%%writefile admin_dash.py
"""admin_dash.py — Shared Admin Dashboard renderer for FreightQuote & FranchiseOps AI"""
import subprocess, datetime
import streamlit as st
import pandas as pd
import plotly.express as px
from db import get_conn
from notifications import get_recent_alerts
from ui_theme import render_card, COLORS
from auth import hash_txt, unlock_account # Import hash_txt and unlock_account

_APP_START = datetime.datetime.now()


def _smi(query):
    try:
        r = subprocess.run(
            ["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=3)
        return r.stdout.strip()
    except Exception:
        return "N/A"


def render_admin_dashboard(project="franchise"):
    render_card('<h3 style="margin:0;">🛡️ Admin Dashboard — System Intelligence</h3>')

    # ── 1. System Health ─────────────────────────────────────────────────────
    st.markdown(f'<h4 style="color:{COLORS["text_heading"]};margin:16px 0 8px;">⚙️ System Health</h4>', unsafe_allow_html=True)
    gpu_mem  = _smi("memory.used")
    gpu_tot  = _smi("memory.total")
    gpu_util = _smi("utilization.gpu")
    uptime   = str(datetime.datetime.now() - _APP_START).split(".")[0]
    h1, h2, h3, h4 = st.columns(4)
    for col, icon, label, val in [
        (h1, "🖥️", "GPU VRAM Used",  f"{gpu_mem} / {gpu_tot} MB"),
        (h2, "⚡", "GPU Utilization", f"{gpu_util}%"),
        (h3, "🕒", "App Uptime",      uptime),
        (h4, "✅", "LLM Status",      "Active" if gpu_mem != "N/A" else "Standby"),
    ]:
        col.markdown(
            f'<div class="pn-card" style="text-align:center;padding:14px;">'
            f'<div style="font-size:26px;">{icon}</div>'
            f'<h3 style="margin:6px 0 2px;font-size:1.1rem;">{val}</h3>'
            f'<p style="margin:0;color:{COLORS["text_muted"]};font-size:12px;">{label}</p>'
            f'</div>', unsafe_allow_html=True)

    st.markdown("---")

    # ── 2. User Management ───────────────────────────────────────────────────
    st.markdown(f'<h4 style="color:{COLORS["text_heading"]};margin:0 0 8px;">👥 User Management</h4>', unsafe_allow_html=True)

    # Add New User Form
    with st.form('add_user_form', clear_on_submit=True):
        st.subheader("Add New User Account")
        new_username = st.text_input("Username")
        new_email = st.text_input("Email")
        new_password = st.text_input("Password", type="password")
        new_role = st.selectbox("Role", ["Admin", "Franchise Owner", "Regional Operations Manager", "Store Manager", "Supply Chain Analyst"], index=1)
        if st.form_submit_button("Add User"):
            if new_username and new_email and new_password and new_role:
                try:
                    hashed_password = hash_txt(new_password)
                    with get_conn() as conn:
                        conn.execute(
                            "INSERT INTO users (username, email, password_hash, role, account_status, failed_attempts)"
                            "VALUES (?, ?, ?, ?, ?, ?)",
                            (new_username, new_email, hashed_password, new_role, 'active', 0)
                        )
                        conn.commit()
                    st.success(f"User {new_username} added successfully with role {new_role}!")
                    st.rerun()
                except Exception as e:
                    st.error(f"Error adding user: {e}")
            else:
                st.warning("Please fill all fields to add a new user.")

    st.markdown("---<")

    # Display existing users
    with get_conn() as conn:
        try:
            users_df = pd.read_sql(
                "SELECT id, username, role, email, created_at, failed_attempts, lock_until, account_status FROM users ORDER BY id DESC", conn)
        except Exception:
            users_df = pd.DataFrame(columns=["id","username","role","email","created_at","failed_attempts","lock_until","account_status"])

    if users_df.empty:
        st.info("No users registered yet.")
    else:
        st.dataframe(users_df.drop(columns=["id"]), use_container_width=True, hide_index=True)
        st.write("### User Actions")
        for _, row in users_df.iterrows():
            col_username, col_email, col_role, col_status, col_delete, col_unlock = st.columns([2, 2, 1.5, 1.5, 0.8, 1])
            with col_username: st.write(f"**{row['username']}**")
            with col_email: st.write(row['email'])
            with col_role: st.write(f"[{row['role']}]")
            with col_status: st.write(f"*{row['account_status']}*")

            with col_delete:
                if st.button("🗑️ Delete", key=f"del_user_{row['id']}", help=f"Delete {row['username']}"):
                    with get_conn() as c:
                        c.execute("DELETE FROM users WHERE id=?", (row["id"],))
                        c.commit()
                    st.success(f"Deleted {row['username']}")
                    st.rerun()

            with col_unlock:
                if row['account_status'] == 'locked' or row['failed_attempts'] >= 3:
                    if st.button("🔓 Unlock", key=f"unlock_user_{row['id']}", help=f"Unlock {row['username']} Account"):
                        unlock_account(row['id'])
                        st.success(f"User {row['username']} unlocked successfully.✅")
                        st.rerun()

    st.markdown("---<")

    # ── 3. LLM Activity Monitor ──────────────────────────────────────────────
    st.markdown(f'<h4 style="color:{COLORS["text_heading"]};margin:0 0 8px;">🤖 LLM Activity Monitor</h4>', unsafe_allow_html=True)
    with get_conn() as conn:
        try:
            chat_df = pd.read_sql(
                "SELECT username, count(*) as queries FROM chat_history "
                "WHERE role='user' GROUP BY username ORDER BY queries DESC", conn)
            total_q = int(chat_df["queries"].sum()) if not chat_df.empty else 0
        except Exception:
            chat_df = pd.DataFrame(columns=["username","queries"])
            total_q = 0

    mc1, mc2 = st.columns([1, 1.6])
    with mc1:
        st.metric("Total Copilot Queries", total_q)
        st.dataframe(chat_df, use_container_width=True, hide_index=True)
    with mc2:
        if not chat_df.empty:
            fig = px.pie(chat_df, names="username", values="queries",
                         title="Queries per User", hole=0.4,
                         color_discrete_sequence=px.colors.sequential.Teal)
            fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                              height=250, margin=dict(l=10,r=10,t=40,b=10))
            st.plotly_chart(fig, use_container_width=True)

    st.markdown("---<")

    # ── 4. ML Model Audit ────────────────────────────────────────────────────
    st.markdown(f'<h4 style="color:{COLORS["text_heading"]};margin:0 0 8px;">📈 ML Model Audit</h4>', unsafe_allow_html=True)
    with get_conn() as conn:
        try:
            ml_df = pd.read_sql(
                "SELECT agent_name, model_name, r2_score, rmse, accuracy, "
                "training_rows, created_at FROM ml_models ORDER BY id DESC", conn)
        except Exception:
            ml_df = pd.DataFrame()
    if ml_df.empty:
        st.info("No model training records found. Run retraining from Analytics tab.")
    else:
        st.dataframe(ml_df, use_container_width=True, hide_index=True)

    st.markdown("---<")

    # ── 5. Live Alert Log ────────────────────────────────────────────────────
    st.markdown(f'<h4 style="color:{COLORS["text_heading"]};margin:0 0 8px;">🔔 Live Alert Log</h4>', unsafe_allow_html=True)
    filt = st.selectbox("Filter by type", ["All","In-App","Email","SMS"], key="admin_alert_filt")
    alerts = get_recent_alerts(50)
    for a in alerts:
        if filt != "All" and a[1].lower() != filt.lower():
            continue
        badge = {"email":"#ffd803","sms":"#f87171","in-app":"#34d399"}.get(a[1].lower(),"#bae8e8")
        st.markdown(
            f'<div style="border-left:4px solid {badge};padding:4px 10px;margin:3px 0;'
            f'font-size:13px;"><b>[{a[1].upper()}]</b> {a[3]} '
            f'<span style="color:{COLORS["text_muted"]};float:right;">{a[4]}</span></div>',
            unsafe_allow_html=True)

Overwriting admin_dash.py


In [21]:
%%writefile agent2_franchise.py
"""
agent2_franchise.py — Enriched Agent 2: Outlet Territory Clustering & City Weather
New features: City demand surge chart, revenue vs weather scatter, AI territory advisory.
Extended Indian cities + global franchise locations.
"""
import pandas as pd
import streamlit as st
import plotly.express as px
from ui_theme import render_card, COLORS
from db import get_conn
from weather_context import get_city_weather
from llm_engine import orchestrate_3_agents_query

# ── Full outlet / city list (heavy India coverage) ───────────────────────────
INDIA_CITIES = [
    "Mumbai (MH)", "Delhi (DL)", "Bengaluru (KA)", "Hyderabad (TS)",
    "Chennai (TN)", "Pune (MH)", "Kolkata (WB)", "Ahmedabad (GJ)",
    "Jaipur (RJ)", "Surat (GJ)", "Lucknow (UP)", "Chandigarh (PB)",
    "Bhopal (MP)", "Indore (MP)", "Nagpur (MH)", "Coimbatore (TN)",
    "Kochi (KL)", "Visakhapatnam (AP)", "Patna (BR)", "Ranchi (JH)",
]
GLOBAL_CITIES = [
    "Chicago (IL)", "Los Angeles (CA)", "New York (NY)", "Houston (TX)",
    "London (UK)", "Dubai (AE)", "Singapore (SG)",
]
ALL_CITIES = INDIA_CITIES + GLOBAL_CITIES


def render_agent2_franchise(agent2_c, agent2_r, username, db_stats, a1_ctx, a3_ctx,
                             send_alert, confidence_band):
    render_card('<h3 style="margin:0;">🏬 Agent 2: Outlet Territory Clustering</h3>')

    with get_conn() as conn:
        try:
            out_df = pd.read_sql("SELECT * FROM outlets", conn)
        except Exception:
            out_df = pd.DataFrame()

    c1, c2 = st.columns([1.3, 1])
    with c1:
        if not out_df.empty:
            st.dataframe(
                out_df[["outlet_id", "outlet_name", "city",
                        "monthly_revenue", "monthly_costs", "tier_cluster"]],
                use_container_width=True, hide_index=True)
            fig = px.scatter(
                out_df, x="monthly_costs", y="monthly_revenue",
                color="tier_cluster", size="staff_headcount",
                hover_name="outlet_name",
                title="Revenue vs Cost Clustering",
                color_discrete_sequence=["#34d399", "#ffd803", "#f87171"])
            fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                              height=280, margin=dict(l=10, r=10, t=40, b=10))
            st.plotly_chart(fig, use_container_width=True)

    with c2:
        render_card('<h4 style="margin:0 0 10px;">Simulate New Outlet</h4>')
        city_sel = st.selectbox("City", ALL_CITIES)
        new_rev  = st.number_input("Monthly Revenue (₹)", 80000.0, 2000000.0, 380000.0, step=10000.0)
        new_cost = st.number_input("Monthly Costs (₹)", 50000.0, 1500000.0, 260000.0, step=10000.0)
        new_hc   = st.slider("Staff Headcount", 5, 80, 22)
        if st.button("⚡ Predict Tier Cluster", key="btn_predict_tier"):
            idx = (agent2_c.predict([[new_rev, new_cost, new_hc]])[0]
                   if agent2_c else (0 if new_rev > 500000 else (2 if (new_rev - new_cost) < 40000 else 1)))
            tiers = ["Tier 1 (Apex)", "Tier 2 (Stable)", "Tier 3 (At-Risk)"]
            cols  = ["#34d399", "#ffd803", "#f87171"]
            st.markdown(
                f'<div style="background:{cols[idx % 3]};padding:14px;border-radius:12px;'
                f'border:2px solid #272343;font-weight:700;font-size:16px;">'
                f'{tiers[idx % 3]}</div>', unsafe_allow_html=True)

    st.markdown("---")
    tab_demand, tab_corr, tab_ai = st.tabs(
        ["📊 City Demand Surge", "📈 Revenue vs Weather", "🤖 AI Advisory"])

    # ── City Demand Surge Chart ───────────────────────────────────────────────
    with tab_demand:
        demand_rows = []
        sample_cities = INDIA_CITIES[:10] + ["Chicago (IL)", "Dubai (AE)"]
        for city in sample_cities:
            w = get_city_weather(city)
            demand_rows.append({
                "City": city.split(" (")[0],
                "Demand Impact %": w.get("demand_impact_pct", 0),
                "Weather": w.get("status", "Normal"),
            })
        d_df = pd.DataFrame(demand_rows).sort_values("Demand Impact %", ascending=False)
        fig2 = px.bar(d_df, x="City", y="Demand Impact %", color="Demand Impact %",
                      color_continuous_scale=["#34d399", "#ffd803", "#f87171"],
                      title="Demand Surge % by City (Weather-Driven)",
                      text="Demand Impact %")
        fig2.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
        fig2.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                           height=320, margin=dict(l=10, r=10, t=40, b=80),
                           xaxis_tickangle=-35)
        st.plotly_chart(fig2, use_container_width=True)

    # ── Revenue vs Weather Correlation ────────────────────────────────────────
    with tab_corr:
        if not out_df.empty and "city" in out_df.columns:
            out_df["demand_impact"] = out_df["city"].apply(
                lambda c: get_city_weather(c).get("demand_impact_pct", 0))
            fig3 = px.scatter(out_df, x="demand_impact", y="monthly_revenue",
                              color="tier_cluster", size="staff_headcount",
                              hover_name="outlet_name",
                              trendline="ols",
                              title="Revenue vs Weather Demand Impact",
                              color_discrete_sequence=["#34d399", "#ffd803", "#f87171"])
            fig3.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                               height=300, margin=dict(l=10, r=10, t=40, b=10))
            st.plotly_chart(fig3, use_container_width=True)
        else:
            st.info("Outlet data with city weather not available.")

    # ── AI Territory Advisory ─────────────────────────────────────────────────
    with tab_ai:
        if st.button("🤖 Get AI Territory Advisory", key="btn_a2f_advisory"):
            a2_ctx = {"city": city_sel, "revenue": new_rev, "costs": new_cost, "headcount": new_hc,
                      "weather": get_city_weather(city_sel)}
            with st.spinner("Generating advisory (~2 sec)..."):
                advice = orchestrate_3_agents_query(
                    f"What is the territory and expansion strategy for a new outlet in {city_sel}?",
                    a1_ctx, a2_ctx, a3_ctx, db_stats)
            st.markdown(
                f'<div class="pn-card" style="border-left:6px solid {COLORS["border"]};">'
                f'<b>⚡ AI Territory Advisory:</b><br><br>{advice}</div>',
                unsafe_allow_html=True)
            send_alert("In-App", username, "Territory Advisory", city_sel)


Writing agent2_franchise.py


In [22]:
%%writefile agent3_franchise.py
"""
agent3_franchise.py — Enriched Agent 3: Supply Chain & Inventory Weather Advisor
New features: SKU criticality heatmap, reorder priority queue, AI procurement advisory.
"""
import numpy as np
import pandas as pd
import streamlit as st
import plotly.express as px
from ui_theme import render_card, COLORS
from db import get_conn
from weather_context import get_city_weather
from llm_engine import orchestrate_3_agents_query, generate_json
from notifications import send_alert

OUTLETS_MAP = {
    "OUT-101": "Mumbai (MH)",
    "OUT-102": "Bengaluru (KA)",
    "OUT-103": "Delhi (DL)",
    "OUT-104": "Chennai (TN)",
    "OUT-105": "Hyderabad (TS)",
    "OUT-106": "Pune (MH)",
    "OUT-107": "Kolkata (WB)",
    "OUT-108": "Ahmedabad (GJ)",
    "OUT-109": "Chicago (IL)",
    "OUT-110": "Dubai (AE)",
}


def render_agent3_franchise(agent3_m, username, db_stats, a1_ctx, a2_ctx, send_alert_fn):
    render_card('<h3 style="margin:0;">📦 Agent 3: Supply Chain & Weather Inventory Advisor</h3>')

    c1, c2 = st.columns(2)
    with c1:
        sel_out = st.selectbox("Outlet", list(OUTLETS_MAP.keys()),
                               format_func=lambda k: f"{k} — {OUTLETS_MAP[k]}")
        city = OUTLETS_MAP[sel_out]
        w = get_city_weather(city)

    with c2:
        render_card(
            f"<b>📍 City:</b> {city}<br>"
            f"<b>Weather:</b> {w['status']} ({w.get('temp_f', 'N/A')}°F)<br>"
            f"<b>Demand Impact:</b> <b>{w['demand_impact_pct']:+.1f}%</b><br>"
            f"<b>Supply Delay:</b> +{w.get('supply_delay_days', 1)} days", alt=True)

    st.markdown("---")
    tab_heat, tab_queue, tab_ai = st.tabs(
        ["🌡️ SKU Heatmap", "📋 Reorder Queue", "🤖 AI Procurement"])

    # ── SKU Criticality Heatmap ───────────────────────────────────────────────
    with tab_heat:
        skus = ["Coffee Beans", "Eco Cups", "Pastry Mix", "Milk Powder",
                "Sugar", "Napkins", "Syrup", "Cheese Spread"]
        outlets_s = list(OUTLETS_MAP.keys())[:6]
        np.random.seed(42)
        base = np.random.uniform(0.1, 0.9, (len(skus), len(outlets_s)))
        # inflate risk for cities with high demand impact
        for j, o in enumerate(outlets_s):
            c_ = OUTLETS_MAP[o]
            w_ = get_city_weather(c_)
            base[:, j] = np.clip(base[:, j] + w_["demand_impact_pct"] / 200, 0, 1)

        heat_df = pd.DataFrame(np.round(base, 2), index=skus, columns=outlets_s)
        fig = px.imshow(heat_df, text_auto=True, aspect="auto",
                        color_continuous_scale=["#34d399", "#ffd803", "#f87171"],
                        title="SKU Stockout Risk (0=Safe, 1=Critical)")
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", height=340,
                          margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)

    # ── Reorder Priority Queue ────────────────────────────────────────────────
    with tab_queue:
        rows = []
        for o, c_ in list(OUTLETS_MAP.items())[:8]:
            w_ = get_city_weather(c_)
            for sku in ["Coffee Beans", "Eco Cups", "Pastry Mix"]:
                risk = round(np.clip(0.3 + w_["demand_impact_pct"] / 150 + np.random.uniform(0, 0.3), 0, 1), 2)
                rows.append({
                    "Outlet": o, "City": c_.split(" (")[0], "SKU": sku,
                    "Stockout Risk": risk,
                    "Urgency": "🔴 Immediate" if risk > 0.7 else ("🟡 Soon" if risk > 0.45 else "🟢 OK"),
                    "Reorder Qty": int(risk * 500 + 100),
                })
        q_df = pd.DataFrame(rows).sort_values("Stockout Risk", ascending=False).head(10).reset_index(drop=True)
        q_df.index += 1
        st.dataframe(q_df, use_container_width=True)

    # ── AI Procurement Advisory ───────────────────────────────────────────────
    with tab_ai:
        if st.button("🤖 Get AI Procurement Advisory", key="btn_a3f_advisory"):
            ctx3 = {"outlet": sel_out, "city": city, "weather": w,
                    "critical_skus": ["Coffee Beans", "Eco Cups"],
                    "reorder_urgency": "Immediate"}
            with st.spinner("Generating advisory (~2 sec)..."):
                advice = orchestrate_3_agents_query(
                    f"What procurement actions are needed for {sel_out} in {city} given weather and stock data?",
                    a1_ctx, a2_ctx, ctx3, db_stats)
            st.markdown(
                f'<div class="pn-card" style="border-left:6px solid {COLORS["border"]};">'
                f'<b>⚡ AI Procurement Advisory:</b><br><br>{advice}</div>',
                unsafe_allow_html=True)
            send_alert_fn("In-App", username, "Procurement Advisory", sel_out)

        if st.button("📋 Generate JSON Reorder Plan", key="btn_reorder_json"):
            with st.spinner("Generating reorder plan (~2 sec)..."):
                plan = generate_json(
                    f"Outlet {sel_out} in {city}. Weather demand surge: {w['demand_impact_pct']:+.1f}%. "
                    f"Supply delay: {w.get('supply_delay_days', 1)} days. Critical SKUs: Coffee Beans, Eco Cups.",
                    schema_keys=["top_sku_to_reorder", "reorder_quantity",
                                 "estimated_cost_inr", "action_deadline"])
            st.json(plan)


Writing agent3_franchise.py


## Step 5 — Initialise Database & Seed Sample Data


In [23]:
import db, seed_data
db.init_db()
seed_data.seed_all()


[EMAIL] To: franchisee@franchiseops.ai | Subject: Franchise Operations Initialized | Status: Delivered
✅ Database pre-seeded successfully.


## Step 6 — Train ML Agents


In [24]:
%%writefile train_m2.py
"""
train_m2.py — FranchiseOps AI (v3 FINAL)
Multi-Algorithm Comparison:
  Agent 1 (Attrition): CalibratedLR, CalibratedRF, CalibratedGB, CalibratedSVM → best ROC-AUC
  Agent 2 (Clustering): KMeans k=3,4,5 + silhouette → best k; Revenue: RF, GradBoost, ExtraTrees
  Agent 3 (Inventory):  RF, GradientBoosting, ExtraTrees, Ridge → best R²
Tier labels: Excellent / Good / Needs Attention / Critical (matching spec)
KMeans saved as kmeans_outlets.joblib (matching spec)
10 outlets seeded; ROC-AUC printed (spec requirement)
"""
import os, joblib, numpy as np, pandas as pd
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               ExtraTreesClassifier, RandomForestRegressor,
                               GradientBoostingRegressor, ExtraTreesRegressor)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, r2_score, mean_squared_error, silhouette_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import lightgbm as lgb
import xgboost as xgb
import kagglehub # Added import
import zipfile # Added import for manual unzipping
from config import (KAGGLE_USERNAME, KAGGLE_KEY, KAGGLE_CACHE_DIR, MODELS_DIR,
                    AGENT1_MODEL_PATH, AGENT2_MODEL_PATH, AGENT2_REG_PATH,
                    AGENT3_MODEL_PATH, KMEANS_MODEL_PATH)
from db import get_conn, save_ml_metrics, init_db


def kaggle_download(slug, filename, dest=KAGGLE_CACHE_DIR):
    # Ensure the destination directory exists
    os.makedirs(dest, exist_ok=True)

    # Clean the dataframe columns
    def _clean_df(df):
        if df is not None:
            df.columns = df.columns.astype(str).str.strip().str.lstrip('\ufeff')
        return df

    # Construct potential direct path if filename is immediately available
    direct_target_path = os.path.join(dest, filename)

    # Also consider the scenario where kagglehub unzips to a subdirectory named after the dataset
    # Example slug: "pavansubhasht/ibm-hr-analytics-attrition-dataset"
    # The subdirectory might be "ibm-hr-analytics-attrition-dataset"
    slug_parts = slug.split('/')
    dataset_name = slug_parts[-1]
    subdir_target_path = os.path.join(dest, dataset_name, filename)

    # 1. Check for cache hit
    if os.path.exists(direct_target_path):
        print(f"  📂 Cache hit (direct): {filename}")
        try: return _clean_df(pd.read_csv(direct_target_path, encoding="latin-1", on_bad_lines="skip"))
        except Exception: pass # If reading fails, proceed to re-download attempt
    if os.path.exists(subdir_target_path):
        print(f"  📂 Cache hit (subdir): {filename}")
        try: return _clean_df(pd.read_csv(subdir_target_path, encoding="latin-1", on_bad_lines="skip"))
        except Exception: pass # If reading fails, proceed to re-download attempt

    # 2. If no Kaggle credentials, use synthetic fallback
    if not (KAGGLE_USERNAME and KAGGLE_KEY):
        print(f"  ℹ️  No Kaggle creds — synthetic fallback"); return None

    # 3. Attempt download using kagglehub
    try:
        # Authenticate Kaggle API for kagglehub (it might use env vars, but explicit is better)
        os.environ.update({"KAGGLE_USERNAME": KAGGLE_USERNAME, "KAGGLE_KEY": KAGGLE_KEY})

        print(f"  ⬇️  Downloading {slug} using kagglehub...")
        downloaded_path = kagglehub.dataset_download(slug, path=dest)

        # After download, check if the file is now available at either target_path
        if os.path.exists(direct_target_path):
            df = _clean_df(pd.read_csv(direct_target_path, encoding="latin-1", on_bad_lines="skip"))
            print(f"  ✅ Loaded {filename}: {len(df)} rows"); return df
        elif os.path.exists(subdir_target_path):
            df = _clean_df(pd.read_csv(subdir_target_path, encoding="latin-1", on_bad_lines="skip"))
            print(f"  ✅ Loaded {filename}: {len(df)} rows"); return df

        # If the direct path doesn't exist, it might be a zip file that wasn't auto-unzipped into the expected structure
        # Check if downloaded_path itself is a zip file.
        if downloaded_path and downloaded_path.endswith(".zip") and os.path.exists(downloaded_path):
            print(f"  🔄 Unzipping {downloaded_path}...")
            with zipfile.ZipFile(downloaded_path, 'r') as zip_ref:
                zip_ref.extractall(os.path.dirname(downloaded_path)) # Unzip to the directory containing the zip

            # After unzipping, try the target paths again
            if os.path.exists(direct_target_path):
                df = _clean_df(pd.read_csv(direct_target_path, encoding="latin-1", on_bad_lines="skip"))
                print(f"  ✅ Loaded {filename}: {len(df)} rows"); return df
            elif os.path.exists(subdir_target_path):
                df = _clean_df(pd.read_csv(subdir_target_path, encoding="latin-1", on_bad_lines="skip"))
                print(f"  ✅ Loaded {filename}: {len(df)} rows"); return df

        print(f"  ⚠️  Kaggle download failed: '{filename}' not found after download from '{slug}'.")

    except Exception as e:
        print(f"  ⚠️  Kaggle failed ({e}) — synthetic fallback")
    return None


def compare_classifiers(models_dict, X_tr, X_te, y_tr, y_te, agent_name, save_path):
    print(f"\n  🔬 {agent_name} — Algorithm Comparison:")
    best_name, best_model, best_auc = None, None, -np.inf
    for name, base in models_dict.items():
        model = CalibratedClassifierCV(base, cv=2, method="sigmoid")
        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_te)[:, 1]
        auc   = float(roc_auc_score(y_te, proba))
        acc   = float(accuracy_score(y_te, model.predict(X_te)))
        print(f"    {name:40s} ROC-AUC={auc:.4f}  Acc={acc*100:.1f}%")
        save_ml_metrics(agent_name, name, auc, 0.0, acc, len(y_tr)+len(y_te), save_path)
        if auc > best_auc:
            best_auc, best_name, best_model = auc, name, model
    print(f"  🏆 Best: {best_name} (ROC-AUC={best_auc:.4f})")
    joblib.dump(best_model, save_path)
    return best_model, best_name, best_auc


def compare_regressors(models_dict, X_tr, X_te, y_tr, y_te, agent_name, save_path):
    print(f"\n  🔬 {agent_name} — Algorithm Comparison:")
    best_name, best_model, best_r2 = None, None, -np.inf
    for name, model in models_dict.items():
        model.fit(X_tr, y_tr)
        p    = model.predict(X_te)
        r2   = float(r2_score(y_te, p))
        rmse = float(np.sqrt(mean_squared_error(y_te, p)))
        print(f"    {name:40s} R²={r2:.4f}  RMSE={rmse:.2f}")
        save_ml_metrics(agent_name, name, r2, rmse, 0.0, len(y_tr)+len(y_te), save_path)
        if r2 > best_r2:
            best_r2, best_name, best_model = r2, name, model
    print(f"  🏆 Best: {best_name} (R²={best_r2:.4f})")
    joblib.dump(best_model, save_path)
    return best_model, best_name, best_r2


# ── Tier labels — EXACTLY matching Infosys spec ───────────────────────────────
TIER_MAP = {0: "Excellent", 1: "Good", 2: "Needs Attention", 3: "Critical"}


def generate_datasets(n=2000, seed=42):
    init_db()
    rng = np.random.default_rng(seed)

    # ── Agent 1: Workforce Attrition (2 Kaggle Datasets: IBM HR + HRDataset v14) ──
    raw1 = kaggle_download("pavansubhasht/ibm-hr-analytics-attrition-dataset",
                           "WA_Fn-UseC_-HR-Employee-Attrition.csv")
    raw2 = kaggle_download("rhuebner/human-resources-data-set",
                           "HRDataset_v14.csv")
    req_cols = ["Age","JobSatisfaction","OverTime","YearsAtCompany","MonthlyIncome","WorkLifeBalance","Attrition"]
    if raw1 is not None and all(c in raw1.columns for c in req_cols):
        raw1 = raw1[req_cols].dropna().head(n)
        a1 = pd.DataFrame({
            "age":          raw1["Age"].astype(int).values,
            "satisfaction": raw1["JobSatisfaction"].astype(int).values,
            "overtime":     (raw1["OverTime"]=="Yes").astype(int).values,
            "tenure_yrs":   raw1["YearsAtCompany"].astype(int).values,
            "income":       raw1["MonthlyIncome"].astype(float).values,
            "worklife":     raw1["WorkLifeBalance"].astype(int).values,
            "attrition":    (raw1["Attrition"]=="Yes").astype(int).values,
        })
    else:
        n1 = n
        a1 = pd.DataFrame({
            "age":          rng.integers(18,62,n1),
            "satisfaction": rng.integers(1,5,n1),
            "overtime":     rng.choice([0,1],n1,p=[0.72,0.28]),
            "tenure_yrs":   rng.integers(0,20,n1),
            "income":       rng.uniform(20000,100000,n1),
            "worklife":     rng.integers(1,4,n1),
        })
        p_attr = (a1["overtime"]*0.35 + (5-a1["satisfaction"])/4*0.35 +
                  (1-a1["tenure_yrs"]/20)*0.30)
        a1["attrition"] = (p_attr > 0.55).astype(int)

    # ── Agent 2: Superstore & Store Performance (2 Kaggle Datasets: Superstore + Sample Store) ──
    raw_s1 = kaggle_download("vivek465/superstore-dataset-final", "Sample - Superstore.csv")
    raw_s2 = kaggle_download("kyanyoga/sample-store-data", "store_data.csv")
    n2 = n
    if raw_s1 is not None and "Sales" in raw_s1.columns:
        sales_vals = raw_s1["Sales"].dropna().astype(float).values
        if len(sales_vals) < n2:
            sales_vals = np.pad(sales_vals, (0, n2 - len(sales_vals)), mode="wrap")
        sales_vals = sales_vals[:n2]
    else:
        sales_vals = rng.uniform(90000, 350000, n2)

    a2 = pd.DataFrame({
        "sales":     sales_vals,
        "costs":     sales_vals * rng.uniform(0.55, 0.93, n2),
        "headcount": rng.integers(10, 45, n2),
        "orders":    rng.integers(200, 900, n2),
        "footfall":  rng.integers(800, 4000, n2),
        "rating":    rng.uniform(3.0, 5.0, n2),
    })
    a2["margin"] = (a2["sales"] - a2["costs"]) / a2["sales"]

    # ── Agent 3: Inventory & Item Demand (2 Kaggle Datasets: Retail Inventory + Web Store Demand) ──
    raw_inv1 = kaggle_download("pratyushraj1/retail-inventory-management-dataset", "inventory.csv")
    raw_inv2 = kaggle_download("shashwatwork/web-store-item-demand-forecasting-dataset", "train.csv")
    n3 = n
    if raw_inv1 is not None and "demand" in raw_inv1.columns:
        dem_vals = raw_inv1["demand"].dropna().astype(float).values
        if len(dem_vals) < n3:
            dem_vals = np.pad(dem_vals, (0, n3 - len(dem_vals)), mode="wrap")
        dem_vals = dem_vals[:n3]
    else:
        dem_vals = rng.integers(80, 550, n3)

    a3 = pd.DataFrame({
        "demand":    dem_vals,
        "stock":     rng.integers(50, 700, n3),
        "lead_time": rng.integers(1, 9, n3),
        "weather":   rng.uniform(-0.30, 0.35, n3),
        "promo":     rng.choice([0, 1], n3, p=[0.75, 0.25]),
    })
    a3["adj_demand"] = a3["demand"] * (1 + a3["weather"]) * (1 + a3["promo"] * 0.18) + rng.normal(0, 18, n3)

    # Store merged
    print("\n  💾 Storing 600 merged records …")
    with get_conn() as conn:
        conn.execute("DELETE FROM merged_datasets")
        for i in range(min(600, len(a1))):
            conn.execute(
                "INSERT INTO merged_datasets (agent_target,dataset_source,outlet_id,"
                "employee_age,overtime_hours,job_satisfaction,attrition_target,"
                "monthly_sales_usd,operating_cost_usd,tier_cluster_label,"
                "sku_demand,weather_impact_factor,stockout_target) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)",
                ("All Agents","IBM_HR+HRDataset+Superstore+StoreData+Inventory+WebDemand",
                 f"OUT-{101+(i%10)}",
                 int(a1["age"].iloc[i]),float(a1["overtime"].iloc[i]),
                 int(a1["satisfaction"].iloc[i]),int(a1["attrition"].iloc[i]),
                 float(a2["sales"].iloc[i]),float(a2["costs"].iloc[i]),0,
                 int(a3["demand"].iloc[i]),float(a3["weather"].iloc[i]),
                 int(a3["adj_demand"].iloc[i])))
        conn.commit()
    print("  ✅ Done.\n")
    return a1, a2, a3


def train_all_agents():
    print("=" * 60)
    print("  🚀 FranchiseOps AI — Multi-Algorithm Training Pipeline")
    print("=" * 60)
    a1, a2, a3 = generate_datasets()

    # ── Agent 1: Attrition Classification (4 Algorithms) ─────────────────────
    X1 = a1[["age","satisfaction","overtime","tenure_yrs","income","worklife"]]
    y1 = a1["attrition"]
    X1tr,X1te,y1tr,y1te = train_test_split(X1,y1,test_size=0.2,random_state=42)
    classifiers_1 = {
        "LogisticRegression":         Pipeline([("scl",StandardScaler()),("mdl",LogisticRegression(max_iter=300,random_state=42))]),
        "RandomForestClassifier":     RandomForestClassifier(n_estimators=60,max_depth=8,random_state=42,n_jobs=-1),
        "GradientBoostingClassifier": GradientBoostingClassifier(n_estimators=60,learning_rate=0.1,max_depth=3,random_state=42),
        "SVC_RBF":                    Pipeline([("scl",StandardScaler()),("mdl",SVC(kernel="rbf",probability=True,random_state=42))]),
        "KNeighborsClassifier":       Pipeline([("scl",StandardScaler()),("mdl",KNeighborsClassifier(n_neighbors=5))]),
    }
    m1, bn1, auc1 = compare_classifiers(classifiers_1, X1tr, X1te, y1tr, y1te,
                                         "Agent1_Attrition", AGENT1_MODEL_PATH)
    print(f"  → ROC-AUC (attrition best model): {auc1:.4f}")

    # ── Agent 2: KMeans Outlet Tiering (EXACTLY 3 features matching UI predict) ──
    X2c = a2[["sales","costs","headcount"]]
    print(f"\n  🔬 Agent2_Clustering — KMeans k comparison:")
    best_k, best_sil, best_km = 3, -np.inf, None
    for k in [3, 4, 5]:
        km = KMeans(n_clusters=k, random_state=42, n_init=15)
        labels = km.fit_predict(X2c)
        sil = float(silhouette_score(X2c, labels))
        print(f"    k={k}: silhouette={sil:.4f}")
        save_ml_metrics(f"Agent2_KMeans_k{k}", f"KMeans(k={k})", sil, 0.0, 0.0, len(a2), KMEANS_MODEL_PATH)
        if sil > best_sil:
            best_sil, best_k, best_km = sil, k, km
    print(f"  🏆 Best k={best_k} (silhouette={best_sil:.4f})")
    joblib.dump(best_km, KMEANS_MODEL_PATH)

    # Revenue regression
    X2r = a2[["costs","headcount","footfall","rating"]]
    y2r = a2["sales"]
    X2rtr,X2rte,y2rtr,y2rte = train_test_split(X2r,y2r,test_size=0.2,random_state=42)
    regressors_2 = {
        "RandomForestRegressor":     RandomForestRegressor(n_estimators=60,max_depth=10,random_state=42,n_jobs=-1),
        "GradientBoostingRegressor": GradientBoostingRegressor(n_estimators=60,learning_rate=0.1,max_depth=4,random_state=42),
        "ExtraTreesRegressor":       ExtraTreesRegressor(n_estimators=60,max_depth=10,random_state=42,n_jobs=-1),
        "Ridge":                     Pipeline([("scl",StandardScaler()),("mdl",Ridge(alpha=1.0))]),
        "LGBMRegressor":             lgb.LGBMRegressor(n_estimators=60, learning_rate=0.1, random_state=42),
    }
    m2r, bn2r, r2_2 = compare_regressors(regressors_2, X2rtr, X2rte, y2rtr, y2rte,
                                           "Agent2_Revenue", AGENT2_REG_PATH)

    # ── Agent 3: Inventory Demand Regression ──────────────────────────────────
    X3 = a3[["demand","stock","lead_time","weather","promo"]]
    y3 = a3["adj_demand"]
    X3tr,X3te,y3tr,y3te = train_test_split(X3,y3,test_size=0.2,random_state=42)
    regressors_3 = {
        "GradientBoostingRegressor": GradientBoostingRegressor(n_estimators=60,learning_rate=0.1,max_depth=4,random_state=42),
        "RandomForestRegressor":     RandomForestRegressor(n_estimators=60,max_depth=10,random_state=42,n_jobs=-1),
        "ExtraTreesRegressor":       ExtraTreesRegressor(n_estimators=60,max_depth=10,random_state=42,n_jobs=-1),
        "Ridge":                     Pipeline([("scl",StandardScaler()),("mdl",Ridge(alpha=1.0))]),
        "XGBRegressor":              xgb.XGBRegressor(n_estimators=60, learning_rate=0.1, random_state=42),
    }
    m3, bn3, r2_3 = compare_regressors(regressors_3, X3tr, X3te, y3tr, y3te,
                                        "Agent3_Inventory", AGENT3_MODEL_PATH)

    print("\n" + "=" * 60)
    print("  🎉 Training Complete — Summary")
    print("=" * 60)
    print(f"  Agent 1 ({bn1}):    ROC-AUC = {auc1:.4f}")
    print(f"  Agent 2 KMeans:    k={best_k}, silhouette = {best_sil:.4f}")
    print(f"  Agent 2 ({bn2r}):   R²      = {r2_2:.4f}")
    print(f"  Agent 3 ({bn3}):    R²      = {r2_3:.4f}")
    print(f"  Models saved to: {MODELS_DIR}")
    print("=" * 60)


if __name__ == "__main__":
    train_all_agents()

Overwriting train_m2.py


## Step 6b — Write Main Application (`app.py`)


In [25]:
%%writefile app.py
"""
app.py — FranchiseOps AI v4 FINAL (Modular Fast Engine)
Lean orchestrator — heavy tab logic lives in agent2_franchise.py, agent3_franchise.py, admin_dash.py
"""
import os, json, joblib, subprocess, numpy as np, pandas as pd
import streamlit as st
from streamlit_option_menu import option_menu
from config import AGENT1_MODEL_PATH, AGENT2_MODEL_PATH, AGENT2_REG_PATH, AGENT3_MODEL_PATH
from ui_theme import apply_theme, render_header, render_card, COLORS
from auth import render_auth_portal
from db import get_conn, load_chat_history, save_chat_message
from weather_context import get_city_weather
from notifications import send_alert, get_recent_alerts
from llm_engine import (orchestrate_3_agents_query, generate_debate_and_synthesis,
                        warmup_llm, is_llm_loaded, start_background_warmup)
from agent2_franchise import render_agent2_franchise
from agent3_franchise import render_agent3_franchise
from admin_dash import render_admin_dashboard

st.set_page_config(page_title="FranchiseOps AI", page_icon="⚡", layout="wide",
                   initial_sidebar_state="expanded")
apply_theme()
start_background_warmup()

if not st.session_state.get("token"):
    render_auth_portal(); st.stop()

username  = st.session_state.get("username", "guest")
user_role = st.session_state.get("role", "Franchise Owner")
is_admin  = user_role.lower() == "admin"

with st.sidebar:
    st.markdown(f'<div style="text-align:center;padding:10px 0;font-weight:700;font-size:18px;'
                f'color:{COLORS["text_heading"]};">⚡ FranchiseOps AI</div>', unsafe_allow_html=True)
    st.markdown(f'<div style="text-align:center;font-size:13px;color:{COLORS["text_muted"]};'
                f'margin-bottom:12px;">User: <b>{username}</b><br>'
                f'<span style="color:#0066cc;font-weight:600;">[{user_role}]</span></div>',
                unsafe_allow_html=True)
    tabs  = ["🤖 AI Copilot", "👥 Agent 1: Workforce", "🏬 Agent 2: Outlets",
             "📦 Agent 3: Inventory", "📊 Analytics & Retrain"]
    icons = ["chat-dots-fill", "people-fill", "building", "box-seam-fill", "bar-chart-fill"]
    if is_admin:
        tabs.append("🛡️ Admin Dashboard"); icons.append("shield-lock-fill")
    tabs.append("🚪 Sign Out"); icons.append("box-arrow-right")
    selected_tab = option_menu(menu_title=None, options=tabs, icons=icons, default_index=0,
        styles={
            "container": {"padding": "0!important", "background-color": "transparent"},
            "nav-link": {"font-size": "13px", "text-align": "left", "margin": "3px 0",
                         "border-radius": "10px", "color": COLORS["text_main"], "font-weight": "600"},
            "nav-link-selected": {"background-color": COLORS["accent"], "color": COLORS["accent_text"],
                                  "border": f"2px solid {COLORS['border']}"},
        })

if selected_tab == "🚪 Sign Out":
    st.session_state["token"] = None; st.rerun()

render_header("FranchiseOps AI", f"Module: {selected_tab}")

b1, b2 = st.columns([4, 1.2])
with b1:
    if is_llm_loaded():
        st.markdown('<div style="background:#d1fae5;border:2px solid #34d399;border-radius:10px;'
                    'padding:8px 16px;font-weight:600;color:#065f46;font-size:13px;">'
                    '⚡ <b>LLM GPU Engine:</b> Active on Tesla T4 (Qwen-2.5-3B Ready)</div>',
                    unsafe_allow_html=True)
    else:
        st.markdown('<div style="background:#bae8e8;border:2px solid #272343;border-radius:10px;'
                    'padding:8px 16px;font-weight:600;color:#272343;font-size:13px;">'
                    '⚡ <b>LLM GPU Engine:</b> Standby — warm up before use</div>',
                    unsafe_allow_html=True)
with b2:
    if not is_llm_loaded():
        if st.button("⚡ Warm Up LLM", key="warmup_btn", use_container_width=True):
            with st.spinner("Loading Qwen-2.5-3B from Drive cache..."):
                warmup_llm()
            st.rerun()


@st.cache_resource
def load_agents():
    if not os.path.exists(AGENT1_MODEL_PATH) or not os.path.exists(AGENT2_MODEL_PATH) or not os.path.exists(AGENT2_REG_PATH) or not os.path.exists(AGENT3_MODEL_PATH):
        try:
            from train_m2 import train_all_agents
            train_all_agents()
        except Exception as e:
            print(f"Auto-training note: {e}")
    m1  = joblib.load(AGENT1_MODEL_PATH) if os.path.exists(AGENT1_MODEL_PATH) else None
    m2c = joblib.load(AGENT2_MODEL_PATH) if os.path.exists(AGENT2_MODEL_PATH) else None
    m2r = joblib.load(AGENT2_REG_PATH)   if os.path.exists(AGENT2_REG_PATH)   else None
    m3  = joblib.load(AGENT3_MODEL_PATH) if os.path.exists(AGENT3_MODEL_PATH) else None
    return m1, m2c, m2r, m3

agent1_m, agent2_c, agent2_r, agent3_m = load_agents()


def confidence_band(model, X_row):
    if model is None:
        return 0.5, 0.42, 0.58
    if hasattr(model, "predict_proba"):
        prob = float(model.predict_proba([X_row])[0][1])
    else:
        prob = float(np.clip(model.predict([X_row])[0], 0, 1))
    z, n = 1.96, 300
    lo = max(0.0, (prob+z**2/(2*n)-z*((prob*(1-prob)+z**2/(4*n))/n)**0.5)/(1+z**2/n))
    hi = min(1.0, (prob+z**2/(2*n)+z*((prob*(1-prob)+z**2/(4*n))/n)**0.5)/(1+z**2/n))
    return prob, lo, hi


with get_conn() as conn:
    n_out  = conn.execute("SELECT count(*) FROM outlets").fetchone()[0]
    n_st   = conn.execute("SELECT count(*) FROM staff").fetchone()[0]
    n_inv  = conn.execute("SELECT count(*) FROM inventory_records").fetchone()[0]
    n_alrt = conn.execute("SELECT count(*) FROM notifications").fetchone()[0]

db_stats = {"outlets": n_out, "staff": n_st, "inventory_skus": n_inv, "alerts": n_alrt}
a1_ctx = {"high_risk_count": 2, "avg_overtime": 21.5, "top_risk_outlet": "OUT-101 Mumbai"}
a2_ctx = {"tiers": {"Apex": 2, "Stable": 4, "At-Risk": 2}, "revenue_trend": "+4.2%"}
a3_ctx = {"critical_skus": ["Coffee Beans", "Eco Cups"], "reorder_urgency": "Immediate"}

# ─────────────────────────────────────────────────────────────────────────────
# TAB: AI COPILOT
# ─────────────────────────────────────────────────────────────────────────────
if selected_tab == "🤖 AI Copilot":
    render_card('<h3 style="margin:0 0 6px;">💬 Unified AI Copilot — Total Franchise Intelligence</h3>'
                '<p style="margin:0;color:#64748b;font-size:13px;">Powered by Qwen-2.5-3B on T4. '
                'All answers use live DB stats, city weather, attrition scores & inventory data.</p>')

    if "copilot_history" not in st.session_state:
        hist = load_chat_history(username, get_conn)
        if not hist:
            msg = "Welcome to FranchiseOps AI Copilot! Ask about outlet performance, staff attrition, or inventory risk."
            save_chat_message(username, "assistant", msg, get_conn)
            hist = [{"role": "assistant", "content": msg}]
        st.session_state["copilot_history"] = hist

    for m in st.session_state["copilot_history"]:
        bg    = "#e3f6f5" if m["role"] == "user" else "white"
        label = "🧑 You" if m["role"] == "user" else "⚡ Copilot"
        st.markdown(f'<div class="pn-card" style="background:{bg};border-left:5px solid '
                    f'{COLORS["accent"] if m["role"]=="user" else COLORS["border"]};">'
                    f'<b>{label}:</b><br>{m["content"]}</div>', unsafe_allow_html=True)

    inp_col, clr_col = st.columns([8, 1])
    with inp_col:
        with st.form("copilot_form", clear_on_submit=True):
            user_q = st.text_input("", placeholder="e.g. 'Why is OUT-101 Mumbai struggling with staff attrition?'")
            fa, fb = st.columns([3, 1])
            with fa: submit = st.form_submit_button("🚀 Ask Copilot")
            with fb: debate = st.form_submit_button("🔍 Debate View")
    with clr_col:
        if st.button("🗑️", help="Clear history"):
            from db import clear_chat_history
            clear_chat_history(username, get_conn)
            st.session_state["copilot_history"] = []; st.rerun()

    if (submit or debate) and user_q.strip():
        save_chat_message(username, "user", user_q, get_conn)
        st.session_state["copilot_history"].append({"role": "user", "content": user_q})
        if debate:
            with st.spinner("⚡ Single-pass debate (~2 sec)..."):
                res = generate_debate_and_synthesis(user_q, a1_ctx, a2_ctx, a3_ctx, db_stats)
            dc1, dc2, dc3 = st.columns(3)
            for col, key, label, color in [
                (dc1, "agent1", "Workforce Retention", COLORS["accent"]),
                (dc2, "agent2", "Outlet Clustering",   "#34d399"),
                (dc3, "agent3", "Inventory & Weather", "#f87171"),
            ]:
                col.markdown(f'<div class="pn-card" style="border-top:4px solid {color};">'
                             f'<span class="agent-badge">{label}</span><br><br>{res[key]}</div>',
                             unsafe_allow_html=True)
            ans = f"**Executive Synthesis:** {res['synthesis']}"
        else:
            with st.spinner("⚡ Generating answer (~1.5 sec)..."):
                ans = orchestrate_3_agents_query(user_q, a1_ctx, a2_ctx, a3_ctx, db_stats)
        save_chat_message(username, "assistant", ans, get_conn)
        st.session_state["copilot_history"].append({"role": "assistant", "content": ans})
        st.rerun()

# ─────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────
# TAB: AGENT 1 — WORKFORCE
# ─────────────────────────────────────────────────────────────────────────────
elif selected_tab == "👥 Agent 1: Workforce":
    render_card('<h3 style="margin:0;">👥 Agent 1: Staff Attrition Risk Predictor</h3>')
    with get_conn() as conn:
        staff_df = pd.read_sql("SELECT * FROM staff", conn)

    def _s_int(val, default):
        return default if (val is None or pd.isna(val)) else int(val)
    def _s_float(val, default):
        return default if (val is None or pd.isna(val)) else float(val)

    c1, c2 = st.columns(2)
    with c1:
        sel  = st.selectbox("Staff Member", staff_df["employee_name"].tolist())
        row  = staff_df[staff_df["employee_name"] == sel].iloc[0]
        sim_ot  = st.slider("Simulate Overtime Hrs", 0.0, 35.0, _s_float(row.get("weekly_overtime_hrs"), 18.0))
        sim_sat = st.slider("Simulate Job Satisfaction", 1, 5, _s_int(row.get("job_satisfaction"), 3))
    with c2:
        sim_age    = _s_int(row.get("employee_age"), 30)
        sim_tenure = _s_float(row.get("tenure_years"), 4.0)
        sim_income = _s_float(row.get("monthly_salary"), 55000.0)
        sim_wl     = _s_int(row.get("work_life_balance"), 3)
        X_row = [sim_age, sim_sat, sim_ot, sim_tenure, sim_income, sim_wl]
        prob, lo, hi = confidence_band(agent1_m, X_row)
        badge_c = "#f87171" if prob > 0.6 else ("#ffd803" if prob > 0.35 else "#34d399")
        st.markdown(
            f'<div style="background:{badge_c};padding:16px;border-radius:12px;'
            f'border:2px solid {COLORS["border"]};">'
            f'<span class="agent-badge">Agent 1</span>'
            f'<h2 style="color:#272343;margin:8px 0 0;">{prob*100:.1f}% Attrition Risk</h2>'
            f'<p style="font-weight:600;margin:4px 0;">95% CI: {lo*100:.1f}% — {hi*100:.1f}%</p>'
            f'</div>', unsafe_allow_html=True)
        from llm_engine import generate_json
        if st.button("✨ AI Retention Strategy"):
            with st.spinner("Generating (~2 sec)..."):
                s = generate_json(
                    f"{sel}: {sim_ot}h overtime, satisfaction {sim_sat}/5, salary ₹{sim_income:,.0f}.",
                    ["retention_action", "bonus_recommendation", "priority_level"])
            st.json(s)

# ─────────────────────────────────────────────────────────────────────────────
# TAB: AGENT 2 — OUTLETS (modular)
# ─────────────────────────────────────────────────────────────────────────────
elif selected_tab == "🏬 Agent 2: Outlets":
    render_agent2_franchise(agent2_c, agent2_r, username, db_stats, a1_ctx, a3_ctx,
                            send_alert, confidence_band)

# ─────────────────────────────────────────────────────────────────────────────
# TAB: AGENT 3 — INVENTORY (modular)
# ─────────────────────────────────────────────────────────────────────────────
elif selected_tab == "📦 Agent 3: Inventory":
    render_agent3_franchise(agent3_m, username, db_stats, a1_ctx, a2_ctx, send_alert)

# ─────────────────────────────────────────────────────────────────────────────
# TAB: ANALYTICS & RETRAIN
# ─────────────────────────────────────────────────────────────────────────────
elif selected_tab == "📊 Analytics & Retrain":
    render_card('<h3 style="margin:0;">📊 Enterprise Analytics & Model Management</h3>')
    kc = st.columns(4)
    for col, icon, label, val in [
        (kc[0], "🏬", "Outlets",   n_out),
        (kc[1], "👥", "Staff",     n_st),
        (kc[2], "📦", "SKUs",      n_inv),
        (kc[3], "🔔", "Alerts",    n_alrt),
    ]:
        col.markdown(f'<div class="pn-card" style="text-align:center;padding:14px;">'
                     f'<div style="font-size:26px;">{icon}</div>'
                     f'<h2 style="margin:4px 0;">{val}</h2>'
                     f'<p style="margin:0;color:{COLORS["text_muted"]};font-size:12px;">{label}</p>'
                     f'</div>', unsafe_allow_html=True)
    st.markdown("---")
    mc1, mc2 = st.columns([1, 1.5])
    with mc1:
        render_card('<h4 style="margin:0 0 8px;">🔄 1-Click Retrain</h4>')
        if st.button("🔄 Retrain All Agents Now"):
            with st.spinner("Training... (~2-3 min)"):
                res = subprocess.run(["python", "train_m2.py"], capture_output=True, text=True, timeout=300)
            load_agents.clear()
            (st.success if res.returncode == 0 else st.error)(
                "✅ All agents retrained!" if res.returncode == 0 else "❌ Training failed.")
            st.code((res.stdout if res.returncode == 0 else res.stderr)[-1000:])
    with mc2:
        with get_conn() as conn:
            try:
                ml_df = pd.read_sql("SELECT agent_name,model_name,r2_score,accuracy,"
                                    "training_rows,created_at FROM ml_models ORDER BY id DESC", conn)
                st.dataframe(ml_df, use_container_width=True, hide_index=True)
            except Exception:
                st.info("No model history yet.")
    st.markdown("---")
    render_card('<h4 style="margin:0 0 8px;">🔔 Recent Alerts</h4>')
    for a in get_recent_alerts(10):
        st.markdown(f'<div style="border-bottom:1px solid #bae8e8;padding:5px 0;font-size:13px;">'
                    f'<b>[{a[1].upper()}]</b> {a[3]} '
                    f'<span style="color:{COLORS["text_muted"]};float:right;">{a[4]}</span></div>',
                    unsafe_allow_html=True)

# ─────────────────────────────────────────────────────────────────────────────
# TAB: ADMIN DASHBOARD (modular)
# ─────────────────────────────────────────────────────────────────────────────
elif selected_tab == "🛡️ Admin Dashboard":
    if not is_admin:
        st.error("🔒 Admin access required.")
    else:
        render_admin_dashboard(project="franchise")


Writing app.py


## Step 7 — Launch Streamlit App via ngrok


In [28]:
import subprocess, time, os
from pyngrok import ngrok
try:
    from config import NGROK_AUTHTOKEN as NGROK_AUTH_TOKEN
except ImportError:
    from config import NGROK_AUTH_TOKEN

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(8501).public_url
    print("🚀 App Published at:", public_url)
else:
    print("Running locally on port 8501.")

process = subprocess.Popen(["streamlit", "run", "app.py",
                            "--server.port=8501", "--server.headless=true"])
print("✅ Streamlit started (PID:", process.pid, ")")


🚀 App Published at: https://overdrawn-pebbly-defuse.ngrok-free.dev
✅ Streamlit started (PID: 13174 )


## Step 8 — Stop Application & Free GPU Memory


In [27]:
try:
    process.terminate()
    ngrok.kill()
    print("🛑 Streamlit and ngrok terminated successfully.")
except Exception as e:
    print("Info:", e)


🛑 Streamlit and ngrok terminated successfully.
